In [1]:
from pyspark.sql import SparkSession

# Initialize Spark Session for local Jupyter development
spark = SparkSession.builder \
    .appName("Subgraph Optimization Log ETL") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "pdm_minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "pdm_minio") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session connected successfully!")

Spark Session connected successfully!


In [2]:
# Define the path to the sample file inside the MinIO bucket
log_file_path = "s3a://datasample/citeseer-ts-32-10-100-1000-triangledensestsubgraph-4.txt"

# Read the file as a raw text dataframe (each line will be a row)
df_logs = spark.read.text(log_file_path)

# Print the total number of lines found in the file
print(f"Total lines in log file: {df_logs.count()}")

# Show the first 10 lines of the file to verify the content
df_logs.show(10, truncate=False)


Total lines in log file: 3102
+-------------------------------------------------------------------------------------------------------------------+
|value                                                                                                              |
+-------------------------------------------------------------------------------------------------------------------+
|FRACTAL_HOME is set to /home/diogo.carvalho3/fractal/bin/..                                                        |
|SPARK_HOME is set to /home/diogo.carvalho3/spark                                                                   |
|app_class is set to 'br.ufmg.cs.systems.fractal.apps.SubgraphOptimizationApp'                                      |
|args is set to '/home/diogo.carvalho3/graphs-data/citeseer 10 100 -1 1000 triangledensestsubgraph ts vertexlabeled'|
|info: FRACTAL_HOME is set to /home/diogo.carvalho3/fractal/bin/..                                                  |
|Description: Script launc

In [3]:
import pyspark.sql.functions as F

# 1. Add a column with the source file name (crucial for processing multiple files later)
df_logs_tagged = df_logs.withColumn("file_path", F.input_file_name())

# 2. Filter to find the line containing the execution arguments
print("--- Extracted Args Line ---")
df_args = df_logs_tagged.filter(F.col("value").contains("args is set to"))
df_args.select("value").show(truncate=False)

# 3. Filter to find the final result line
print("--- Extracted Best Subgraph Line ---")
df_best = df_logs_tagged.filter(F.col("value").contains("BestSubgraph="))
df_best.select("value").show(truncate=False)

--- Extracted Args Line ---
+-------------------------------------------------------------------------------------------------------------------+
|value                                                                                                              |
+-------------------------------------------------------------------------------------------------------------------+
|args is set to '/home/diogo.carvalho3/graphs-data/citeseer 10 100 -1 1000 triangledensestsubgraph ts vertexlabeled'|
+-------------------------------------------------------------------------------------------------------------------+

--- Extracted Best Subgraph Line ---
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                           

In [4]:
import pyspark.sql.functions as F

# Define the Regex patterns
repetition_regex = r"-(\d+)\.txt$"
args_regex = r"args is set to '.*?/([^/ ]+)\s+(\d+)\s+(\d+)\s+(?:-?\d+)\s+(\d+)\s+([a-zA-Z0-9_]+)\s+([a-zA-Z0-9_]+)"
threads_regex = r"--executor-cores\s+(\d+)"

# BestSubgraph line patterns
time_regex = r"ElapsedTimeMs=(\d+)"
cost_regex = r"BestSubgraph=.*cost=([0-9.]+)"
v_regex = r"BestSubgraph=.*nvertices=(\d+)"
e_regex = r"BestSubgraph=.*nedges=(\d+)"

# Apply the Regex to every line in the DataFrame
df_extracted = df_logs_tagged.select(
    "file_path",
    F.regexp_extract("file_path", repetition_regex, 1).cast("int").alias("repetition"),
    
    F.regexp_extract("value", args_regex, 1).alias("graph_name"),
    F.regexp_extract("value", args_regex, 6).alias("metaheuristic"),
    F.regexp_extract("value", args_regex, 2).cast("int").alias("initial_vertices"),
    F.regexp_extract("value", args_regex, 3).cast("int").alias("num_initial_solutions"),
    F.regexp_extract("value", args_regex, 4).cast("int").alias("timeout_ms"),
    F.regexp_extract("value", args_regex, 5).alias("objective_function"),
    
    F.regexp_extract("value", threads_regex, 1).cast("int").alias("num_threads"),
    
    F.regexp_extract("value", time_regex, 1).cast("int").alias("total_time_ms"), # <--- NEW
    F.regexp_extract("value", cost_regex, 1).cast("double").alias("cost_best_solution"),
    F.regexp_extract("value", v_regex, 1).cast("int").alias("vertices_best_solution"),
    F.regexp_extract("value", e_regex, 1).cast("int").alias("edges_best_solution")
)

# Clean empty strings into Nulls
def blank_as_null(col_name):
    return F.when(F.col(col_name) != "", F.col(col_name)).otherwise(F.lit(None))

df_cleaned = df_extracted.select(
    "file_path", "repetition", "initial_vertices", "num_initial_solutions", 
    "timeout_ms", "num_threads", "total_time_ms", "cost_best_solution", 
    "vertices_best_solution", "edges_best_solution",
    blank_as_null("graph_name").alias("graph_name"),
    blank_as_null("metaheuristic").alias("metaheuristic"),
    blank_as_null("objective_function").alias("objective_function")
)

# Group by the File and squash into ONE row using max()
df_final_parameters = df_cleaned.groupBy("file_path", "repetition").agg(
    F.max("graph_name").alias("graph_name"),
    F.max("metaheuristic").alias("metaheuristic"),
    F.max("initial_vertices").alias("initial_vertices"),
    F.max("num_initial_solutions").alias("num_initial_solutions"),
    F.max("timeout_ms").alias("timeout_ms"),
    F.max("objective_function").alias("objective_function"),
    F.max("num_threads").alias("num_threads"),
    F.max("total_time_ms").alias("total_time_ms"), # <--- NEW
    F.max("cost_best_solution").alias("cost_best_solution"),
    F.max("vertices_best_solution").alias("vertices_best_solution"),
    F.max("edges_best_solution").alias("edges_best_solution")
)

# Show the squashed result!
df_final_parameters.drop("file_path").show(truncate=False)

+----------+----------+-------------+----------------+---------------------+----------+-----------------------+-----------+-------------+------------------+----------------------+-------------------+
|repetition|graph_name|metaheuristic|initial_vertices|num_initial_solutions|timeout_ms|objective_function     |num_threads|total_time_ms|cost_best_solution|vertices_best_solution|edges_best_solution|
+----------+----------+-------------+----------------+---------------------+----------+-----------------------+-----------+-------------+------------------+----------------------+-------------------+
|4         |citeseer  |ts           |10              |100                  |1000      |triangledensestsubgraph|32         |2227         |23.333333333333332|18                    |84                 |
+----------+----------+-------------+----------------+---------------------+----------+-----------------------+-----------+-------------+------------------+----------------------+-------------------+


In [5]:
import pyspark.sql.functions as F

# --- 1. Define all Regex Patterns ---
repetition_regex = r"-(\d+)\.txt$"
args_regex = r"args is set to '.*?/([^/ ]+)\s+(\d+)\s+(\d+)\s+(?:-?\d+)\s+(\d+)\s+([a-zA-Z0-9_]+)\s+([a-zA-Z0-9_]+)"
threads_regex = r"--executor-cores\s+(\d+)"

# Final Result Metrics
time_regex = r"ElapsedTimeMs=(\d+)"
cost_regex = r"BestSubgraph=.*cost=([0-9.]+)"
v_regex = r"BestSubgraph=.*nvertices=(\d+)"
e_regex = r"BestSubgraph=.*nedges=(\d+)"

# NEW: Run number, Timestamps, and Intermediate values
run_regex = r"SubgraphOptimization\s+(\d+)"
timestamp_regex = r"^(\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})"

# Extracts from: "26/01/30 23:50:56 ClassName run nvertices nedges ... cost"
inter_regex = r"^\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}\s+\S+\s+\d+\s+(\d+)\s+(\d+).*\s([0-9.]+)$"

# --- 2. Extract Everything into Columns ---
df_extracted = df_logs_tagged.select(
    "file_path",
    "value",
    F.regexp_extract("file_path", repetition_regex, 1).cast("int").alias("repetition"),
    
    # Original parameters
    F.regexp_extract("value", args_regex, 1).alias("graph_name"),
    F.regexp_extract("value", args_regex, 6).alias("metaheuristic"),
    F.regexp_extract("value", args_regex, 2).cast("int").alias("initial_vertices"),
    F.regexp_extract("value", args_regex, 3).cast("int").alias("num_initial_solutions"),
    F.regexp_extract("value", args_regex, 4).cast("int").alias("timeout_ms"),
    F.regexp_extract("value", args_regex, 5).alias("objective_function"),
    F.regexp_extract("value", threads_regex, 1).cast("int").alias("num_threads"),
    
    # Final best results
    F.regexp_extract("value", time_regex, 1).cast("int").alias("total_time_ms"),
    F.regexp_extract("value", cost_regex, 1).cast("double").alias("cost_best_solution"),
    F.regexp_extract("value", v_regex, 1).cast("int").alias("vertices_best_solution"),
    F.regexp_extract("value", e_regex, 1).cast("int").alias("edges_best_solution"),
    
    # NEW: effective_runs & timestamps
    F.regexp_extract("value", run_regex, 1).cast("int").alias("run_number"),
    F.to_timestamp(F.regexp_extract("value", timestamp_regex, 1), "yy/MM/dd HH:mm:ss").alias("log_timestamp"),
    
    # NEW: Intermediate states
    F.regexp_extract("value", inter_regex, 1).cast("int").alias("inter_v"),
    F.regexp_extract("value", inter_regex, 2).cast("int").alias("inter_e"),
    F.regexp_extract("value", inter_regex, 3).cast("double").alias("inter_cost")
)

# --- 3. First Aggregation: Get the Final Parameters ---
df_final_parameters = df_extracted.groupBy("file_path", "repetition").agg(
    F.max("graph_name").alias("graph_name"),
    F.max("metaheuristic").alias("metaheuristic"),
    F.max("initial_vertices").alias("initial_vertices"),
    F.max("num_initial_solutions").alias("num_initial_solutions"),
    F.max("timeout_ms").alias("timeout_ms"),
    F.max("objective_function").alias("objective_function"),
    F.max("num_threads").alias("num_threads"),
    F.max("total_time_ms").alias("total_time_ms"),
    F.max("cost_best_solution").alias("cost_best_solution"),
    F.max("vertices_best_solution").alias("vertices_best_solution"),
    F.max("edges_best_solution").alias("edges_best_solution"),
    
    # Calculate effective runs (+1 because it starts at 0)
    (F.max("run_number") + 1).alias("effective_runs"),
    
    # Get the absolute start time of the file
    F.min("log_timestamp").alias("start_timestamp")
)

# The Temporal Join: Find the Time to Best Solution
# CREATE A LEAN TIMELINE TABLE: Select ONLY the intermediate columns to prevent duplicate names
df_timeline = df_extracted.filter(F.col("inter_cost").isNotNull()).select(
    "file_path", "log_timestamp", "inter_v", "inter_e", "inter_cost"
)

# Join the lean timeline with our perfect final parameters
df_join = df_timeline.join(
    F.broadcast(df_final_parameters),
    on="file_path",
    how="inner"
)

# Filter for the line where the intermediate metrics perfectly match the final best metrics
df_matched = df_join.filter(
    (F.col("inter_v") == F.col("vertices_best_solution")) &
    (F.col("inter_e") == F.col("edges_best_solution")) &
    (F.abs(F.col("inter_cost") - F.col("cost_best_solution")) < 1e-6)
)

# Find the earliest timestamp where this perfect match occurred
df_best_time = df_matched.groupBy("file_path").agg(
    F.min("log_timestamp").alias("best_solution_timestamp")
)

# --- 5. Final Calculation ---
# Join the best time back to the main table
df_final_table = df_final_parameters.join(df_best_time, on="file_path", how="left")

# Calculate the difference in milliseconds
df_final_table = df_final_table.withColumn(
    "time_to_best_solution_ms",
    (F.unix_timestamp("best_solution_timestamp") - F.unix_timestamp("start_timestamp")) * 1000
)

# Drop temporary columns to clean up the final table
df_final_table = df_final_table.drop("start_timestamp", "best_solution_timestamp", "file_path")

# Show the ultimate result!
df_final_table.show(truncate=False)

+----------+----------+-------------+----------------+---------------------+----------+-----------------------+-----------+-------------+------------------+----------------------+-------------------+--------------+------------------------+
|repetition|graph_name|metaheuristic|initial_vertices|num_initial_solutions|timeout_ms|objective_function     |num_threads|total_time_ms|cost_best_solution|vertices_best_solution|edges_best_solution|effective_runs|time_to_best_solution_ms|
+----------+----------+-------------+----------------+---------------------+----------+-----------------------+-----------+-------------+------------------+----------------------+-------------------+--------------+------------------------+
|4         |citeseer  |ts           |10              |100                  |1000      |triangledensestsubgraph|32         |2227         |23.333333333333332|18                    |84                 |50            |0                       |
+----------+----------+-------------+---

In [6]:
# 1. Let's look at the raw lines that matched our timeline Regex
print("--- Extracted Timeline (Top 20 rows) ---")
# Use df_extracted directly (it still has 'value')
df_debug = df_extracted.filter(F.col("inter_cost").isNotNull())

# Now select all needed columns, including the raw 'value'
df_debug.select("log_timestamp", "inter_v", "inter_e", "inter_cost", "value") \
        .show(20, truncate=False)
df_debug.select("log_timestamp", "inter_v", "inter_e", "inter_cost", df_extracted["value"]).show(20, truncate=False)

# 2. Let's explicitly search the timeline for our target cost to see if it even exists in the file!
print("\n--- Searching for Cost ≈ 20.769 ---")
df_debug.filter(F.abs(F.col("inter_cost") - 20.76923076923077) < 0.1).show(truncate=False)

--- Extracted Timeline (Top 20 rows) ---
+-------------------+-------+-------+----------+-----------------------------------------------------------------------------------------------------+
|log_timestamp      |inter_v|inter_e|inter_cost|value                                                                                                |
+-------------------+-------+-------+----------+-----------------------------------------------------------------------------------------------------+
|2026-02-02 18:47:53|10     |16     |1.2       |26/02/02 18:47:53 SubgraphOptimization 10 10 16 3228,3234,3267,3271,3270,3268,3290,3250,3240,3252 1.2|
|2026-02-02 18:47:53|10     |14     |1.8       |26/02/02 18:47:53 SubgraphOptimization 21 10 14 2856,3198,3019,3018,2838,3269,2888,2678,3014,2854 1.8|
|2026-02-02 18:47:53|10     |9      |0.0       |26/02/02 18:47:53 SubgraphOptimization 24 10 9 2792,740,1817,2694,729,2356,2524,3209,2723,1974 0.0   |
|2026-02-02 18:47:53|10     |9      |0.0       |26/02